### Imports

In [9]:
import pydantic
import langchain_core

print("pydantic:", pydantic.__version__)
print("pydantic path:", pydantic.__file__)

print("langchain_core:", langchain_core.__version__)
print("langchain_core path:", langchain_core.__file__)

pydantic: 2.13.4
pydantic path: f:\PANTA\Projects\Banking-FAQ-RAG-System\banking_ai_assistant_venv\Lib\site-packages\pydantic\__init__.py
langchain_core: 1.4.1
langchain_core path: f:\PANTA\Projects\Banking-FAQ-RAG-System\banking_ai_assistant_venv\Lib\site-packages\langchain_core\__init__.py


In [10]:
import sys
print(sys.executable)

f:\PANTA\Projects\Banking-FAQ-RAG-System\banking_ai_assistant_venv\python.exe


In [1]:
import os
import re
import pickle
import numpy as np
import pandas as pd

from dotenv import load_dotenv

from sklearn.metrics.pairwise import cosine_similarity

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

from scipy.sparse import hstack

### Load Environment Variables

In [2]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

### Load Intent Classification Artifacts

In [3]:
with open("../models/intent_classification/best_intent_classifier.pkl", "rb") as f:
    best_model = pickle.load(f)

with open("../models/intent_classification/word_vectorizer.pkl", "rb") as f:
    word_vectorizer = pickle.load(f)

with open("../models/intent_classification/char_vectorizer.pkl", "rb") as f:
    char_vectorizer = pickle.load(f)

with open("../models/intent_classification/label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

### Load Processed Dataset

In [4]:
df = pd.read_csv("../data/processed_data/02_banking_processed.csv")

print(df.shape)

(2331, 9)


### Load Embedding Model

In [5]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

### Create Banking Reference Embeddings

Instead of embedding all rows every query: Create once.

In [6]:
banking_questions = (df["processed_question"].dropna().tolist())
banking_embeddings = embeddings.embed_documents(banking_questions)
banking_embeddings = np.array(banking_embeddings)

banking_embeddings.shape

(2331, 384)

### Save Banking Embeddings

In [7]:
os.makedirs("../models/relevance", exist_ok=True)

with open("../models/relevance/banking_reference_embeddings.pkl", "wb") as f:
    pickle.dump(banking_embeddings, f)

### Query Preprocessing

In [8]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

### Intent Confidence Check

In [9]:
def get_intent_confidence(question):
    clean_question = preprocess_text(question)
    word_features = word_vectorizer.transform([clean_question])
    char_features = char_vectorizer.transform([clean_question])
    features = hstack([word_features, char_features])
    prediction = best_model.predict(features)
    predicted_label = (label_encoder.inverse_transform(prediction)[0])
    confidence = (best_model.predict_proba(features).max() * 100)

    return (predicted_label, confidence)

### Semantic Similarity Check

In [10]:
def get_similarity_score(question):
    query_embedding = embeddings.embed_query(question)
    similarities = cosine_similarity([query_embedding], banking_embeddings)[0]

    return similarities.max()

#### Test Similarity

In [11]:
get_similarity_score("How can I block my ATM card?")

0.8045074034815869

In [12]:
get_similarity_score("Who won IPL final?")

0.292192532152785

### LLM Relevance Judge

In [13]:
RELEVANCE_PROMPT = """
You are a Banking Query Validator.

Determine whether the user's query belongs to the banking domain.

Valid Banking Topics:

- Savings Accounts
- Current Accounts
- Fixed Deposits
- Loans
- Credit Cards
- Debit Cards
- UPI
- RTGS
- NEFT
- Banking Security
- Banking Fraud
- Mutual Funds
- Insurance
- Banking Regulations
- Banking Customer Support

Return ONLY:

Relevant

or

Not Relevant

Query:

{query}
"""

### Load Judge LLM

In [ ]:
judge_llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.1-8b-instant",
    temperature=0
)

#### LLM Judge Function

In [15]:
def llm_relevance_check(query):
    prompt = RELEVANCE_PROMPT.format(query=query)
    response = judge_llm.invoke(prompt)

    return response.content.strip()

### Enterprise Relevance Engine

In [16]:
def knowledge_base_relevance_check(question):
    predicted_category, confidence = (get_intent_confidence(question))
    similarity_score = (get_similarity_score(question))
    llm_decision = (llm_relevance_check(question))

    result = {
        "question": question,
        "predicted_category": predicted_category,
        "intent_confidence": round(confidence, 2),
        "semantic_similarity": round(similarity_score, 4),
        "llm_decision": llm_decision
    }

    if (confidence >= 50 and similarity_score >= 0.55 and llm_decision == "Relevant"):
        result["final_decision"] = "Relevant"

    elif similarity_score >= 0.40:
        result["final_decision"] = ("Partially Relevant")
    else:
        result["final_decision"] = ("Not Relevant")

    return result

#### Banking Query Tests

In [19]:
knowledge_base_relevance_check("How do I activate UPI?")

{'question': 'How do I activate UPI?',
 'predicted_category': 'Digital & Security',
 'intent_confidence': 35.21,
 'semantic_similarity': 0.691,
 'llm_decision': 'Relevant',
 'final_decision': 'Partially Relevant'}

In [20]:
knowledge_base_relevance_check("How can I apply for home loan?")

{'question': 'How can I apply for home loan?',
 'predicted_category': 'Loans',
 'intent_confidence': 64.99,
 'semantic_similarity': 0.7806,
 'llm_decision': 'Relevant',
 'final_decision': 'Relevant'}

In [21]:
knowledge_base_relevance_check("How can I apply for home loan?")

{'question': 'How can I apply for home loan?',
 'predicted_category': 'Loans',
 'intent_confidence': 64.99,
 'semantic_similarity': 0.7806,
 'llm_decision': 'Relevant',
 'final_decision': 'Relevant'}

#### Non-Banking Query Tests

In [22]:
knowledge_base_relevance_check("Who won IPL 2025?")

{'question': 'Who won IPL 2025?',
 'predicted_category': 'Investments & Insurance',
 'intent_confidence': 43.95,
 'semantic_similarity': 0.2909,
 'llm_decision': 'Not Relevant',
 'final_decision': 'Not Relevant'}

In [23]:
knowledge_base_relevance_check("Tell me a joke.")

{'question': 'Tell me a joke.',
 'predicted_category': 'Investments & Insurance',
 'intent_confidence': 47.54,
 'semantic_similarity': 0.2406,
 'llm_decision': 'Not Relevant',
 'final_decision': 'Not Relevant'}

### Batch Validation

In [24]:
test_queries = [
    "How can I open a savings account?",
    "What is RTGS transfer?",
    "Who won IPL?",
    "Tell me a joke",
    "How do I reset ATM PIN?"
]

In [26]:
results = []

for query in test_queries:
    results.append(knowledge_base_relevance_check(query))

pd.DataFrame(results)

,question,predicted_category,intent_confidence,semantic_similarity,llm_decision,final_decision
0,How can I open a savings account?,Retail Banking,89.62,0.7563,Relevant,Relevant
1,What is RTGS transfer?,Customer Support,32.91,0.8674,Relevant,Partially Relevant
2,Who won IPL?,Investments & Insurance,43.95,0.3023,Not Relevant,Not Relevant
3,Tell me a joke,Investments & Insurance,47.54,0.2694,Not Relevant,Not Relevant
4,How do I reset ATM PIN?,Cards & Payments,52.74,0.8501,Relevant,Relevant


### Save Relevance Configuration

In [28]:
relevance_config = {
    "intent_threshold": 50,
    "similarity_threshold": 0.55,
    "embedding_model": "all-MiniLM-L6-v2",
    "judge_model": "llama-3.3-70b-versatile"
}

with open("../models/relevance/relevance_config.pkl", "wb") as f:
    pickle.dump(relevance_config, f)

## Key Insights

## Knowledge Base Relevance Layer Created

Implemented:

User Query
↓
Intent Confidence Check
↓
Embedding Similarity Check
↓
LLM Relevance Judge
↓
Final Relevance Decision

---

## Validation Layers

### Layer 1

Intent Classification Confidence

Purpose:

- Check whether query resembles known banking categories

---

### Layer 2

Semantic Similarity

Purpose:

- Compare query against entire banking knowledge base
- Reject unrelated questions

---

### Layer 3

LLM Judge

Purpose:

- Handle unseen banking questions
- Detect domain relevance using reasoning

---

## Final Decisions

Possible outputs:

- Relevant
- Partially Relevant
- Not Relevant

---

## Enterprise Benefit

This layer prevents:

- Out-of-domain questions
- Irrelevant retrieval
- Unnecessary LLM calls
- Banking policy violations

---

## Next Notebook

10_response_validation.ipynb

We will implement:

- Groundedness Check
- Hallucination Detection
- Evidence Validation
- Trust Score Calculation

before any response reaches the user.